# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login

login()

In [2]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [13]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, duckdb, os
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs_v2"

# Load the base Feb-Apr feature/label data
data_model = pd.read_csv(f"{BASE}/data_model_v2.csv")
print("Loaded:", data_model.shape)
print(data_model.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (183345, 23)
['client_hash_id', 'content_hash_id', 'impressions_window', 'clicks_window', 'april_impressions', 'april_clicks', 'february_clicks', 'click_through_rate', 'weighted_position', 'momentum', 'active_days', 'click_through_rate_missing', 'weighted_position_missing', 'momentum_missing', 'clicks_april', 'clicks_may', 'declined', 'momentum_risk', 'baseline_score', 'reason_code', 'action', 'content_age_days', 'content_age_days_missing']


In [15]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_arr = y_true.values if hasattr(y_true, "values") else np.asarray(y_true)
    return y_arr[order[:k]].mean()

print("Setup complete:", data_model.shape)

Setup complete: (183345, 23)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression first, then Random Forest. My lane's question shape is "which pages first?" — a ranking problem with an observed label (declined). Per the toolkit, that's the "yes/no with an observed label" row → start readable (Logistic Regression), then check if a stronger model (Random Forest) earns its added complexity. Since the deliverable is a ranked queue, evaluation uses precision@50 — matching the baseline's own metric — not just raw accuracy or AUC alone, though AUC is reported too since it captures overall ranking quality across the full portfolio, not just the top 50.

In [16]:
method_choice = "Logistic Regression -> Random Forest, evaluated at precision@50 (matching baseline)"
print(method_choice)

Logistic Regression -> Random Forest, evaluated at precision@50 (matching baseline)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_hash_id. Confirmed necessary in Week 6: a naive random split allowed 49 clients to overlap between train and test, inflating precision@50 to a perfect 100% — a clear memorization artifact, not real skill. A grouped split tests the honest question: does this model work on a client it has never seen any pages from? This matters especially given the concentration finding (top 5 clients = 56.6% of the portfolio) — without grouping, the model could simply learn to recognize dominant clients rather than a generalizable decline pattern.

In [17]:
from sklearn.model_selection import GroupShuffleSplit

model_cols = ["impressions_window", "clicks_window", "april_impressions", "april_clicks",
              "february_clicks", "click_through_rate", "weighted_position", "momentum",
              "active_days", "click_through_rate_missing", "weighted_position_missing", "momentum_missing"]

X = data_model[model_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

overlap = len(set(data_model.iloc[train_idx]["client_hash_id"]) & set(data_model.iloc[test_idx]["client_hash_id"]))
print("Client overlap (must be 0):", overlap)
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))

Client overlap (must be 0): 0
Train rows: 154281 | Test rows: 29064


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_arr = y_true.values if hasattr(y_true, "values") else np.asarray(y_true)
    return y_arr[order[:k]].mean()

y_test = y.iloc[test_idx].reset_index(drop=True)

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
logreg_scores = logreg.predict_proba(X.iloc[test_idx])[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
rf_scores = rf.predict_proba(X.iloc[test_idx])[:, 1]

# Baseline: impressions_window, on the SAME test rows
baseline_scores_test = data_model.iloc[test_idx]["impressions_window"].reset_index(drop=True).values

comparison = pd.DataFrame({
    "method": ["Base rate", "Week-4 baseline (volume)", "Logistic Regression", "Random Forest"],
    "precision@50": [
        y_test.mean(),
        precision_at_k(y_test, baseline_scores_test, 50),
        precision_at_k(y_test, logreg_scores, 50),
        precision_at_k(y_test, rf_scores, 50),
    ],
    "AUC": [
        0.5,
        roc_auc_score(y_test, baseline_scores_test),
        roc_auc_score(y_test, logreg_scores),
        roc_auc_score(y_test, rf_scores),
    ]
})
comparison

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,method,precision@50,AUC
0,Base rate,0.18879,0.500000
1,Week-4 baseline (volume),0.56000,0.785699
2,Logistic Regression,0.46000,0.867310
3,Random Forest,0.54000,0.930229


The comparison reveals a genuine, non-obvious finding: Random Forest achieves substantially higher overall AUC (0.930 vs. 0.786) than the simple volume-based baseline, indicating better ranking quality across the full portfolio. However, at precision@50 — the metric matching this lane's actual weekly review capacity — the baseline (56%) outperforms both Random Forest (54%) and Logistic Regression (46%). This suggests the models capture more generalizable, nuanced risk patterns overall, but the baseline's brute-force "biggest pages first" logic is extremely hard to beat at the very top of the queue, where sheer volume is nearly a perfect proxy for "room to decline." Per the training-honest-models standard, both results are reported rather than picking the more flattering one.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [19]:
importances = pd.Series(rf.feature_importances_, index=model_cols).sort_values(ascending=False)
print(importances)
top_share = importances.iloc[0]
print(f"\nTop feature: {importances.index[0]} ({top_share:.1%})",
      "— investigate, suspiciously high" if top_share > 0.7 else "— no single feature dominates")

april_clicks                  0.419269
click_through_rate            0.181487
clicks_window                 0.120373
weighted_position_missing     0.119007
momentum                      0.065228
april_impressions             0.042163
impressions_window            0.024558
february_clicks               0.008913
weighted_position             0.007670
active_days                   0.006805
momentum_missing              0.004528
click_through_rate_missing    0.000000
dtype: float64

Top feature: april_clicks (41.9%) — no single feature dominates


In [ ]:
test_results = data_model.iloc[test_idx].copy().reset_index(drop=True)
test_results["true_label"] = y_test
test_results["rf_score"] = rf_scores

false_negatives = test_results[test_results["true_label"] == 1].sort_values("rf_score").head(3)
print("Worst false negatives (missed real declines):")
print(false_negatives[["client_hash_id", "content_hash_id", "impressions_window", "click_through_rate", "rf_score"]])

false_positives = test_results[test_results["true_label"] == 0].sort_values("rf_score", ascending=False).head(3)
print("\nWorst false positives (flagged but didn't decline):")
print(false_positives[["client_hash_id", "content_hash_id", "impressions_window", "click_through_rate", "rf_score"]])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.